In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

HF_TOKEN = os.getenv('HF_TOKEN')
from huggingface_hub import login
login(HF_TOKEN)

c:\Users\theke\Documents\Code\bharatanatyam-mudra-classification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [2]:
from datasets import load_dataset

dataset = load_dataset("Samarth0710/bharatanatyam-mudra-dataset", split="train")
# Just single-hand
dataset = dataset.filter(lambda row: row["gesture_type"] == "single_hand")

labels = sorted(set(dataset["label"]))
label_to_id = {l: i for i, l in enumerate(labels)}
dataset

Dataset({
    features: ['image', 'label', 'gesture_type', 'label_id'],
    num_rows: 14827
})

In [3]:
import numpy as np
import mediapipe as mp
from mediapipe.tasks.python import vision
from mediapipe.tasks import python as mp_python
from datasets import load_dataset
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm
import cv2

In [4]:
MODEL_PATH = "googletutorial/hand_landmarker.task"
base_options = mp_python.BaseOptions(model_asset_path=MODEL_PATH)
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_hands=2
)
detector = vision.HandLandmarker.create_from_options(options)

In [5]:
IMAGE_SIZE   = (224, 224)
PADDING      = 0.2
BATCH_SIZE   = 32
EPOCHS       = 50
LEARNING_RATE = 1e-3
FINE_TUNE_LR  = 1e-5
FINE_TUNE_EPOCH_START = 10

In [6]:
def get_bounding_box(hand_landmarks, img_w, img_h, padding=PADDING):
    xs = [lm.x * img_w for lm in hand_landmarks]
    ys = [lm.y * img_h for lm in hand_landmarks]
    x1 = max(0, int(min(xs) - padding * img_w))
    x2 = min(img_w, int(max(xs) + padding * img_w))
    y1 = max(0, int(min(ys) - padding * img_h))
    y2 = min(img_h, int(max(ys) + padding * img_h))
    return x1, y1, x2, y2


def normalize_hand(hand_landmarks):
    """no rotation, to stay consistent with image."""
    pts = np.array([[lm.x, lm.y] for lm in hand_landmarks])  # (21, 2)
    pts -= pts[0]
    scale = np.linalg.norm(pts[9])
    if scale > 1e-6:
        pts /= scale
    return pts.flatten()  

def extract_model_inputs(pil_image):
    """
    Returns
    -------
        cropped_image: (224, 224, 3) uint8 numpy array, or None
        landmarks:     (84,) float32 numpy array, or None
    """
    img_rgb = np.array(pil_image.convert("RGB"))
    h, w = img_rgb.shape[:2]

    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
    results = detector.detect(mp_image)

    if not results.hand_landmarks:
        return None, None

    # Landmarks (raw, position+scale normalized)
    landmark_feats = []
    for hand in results.hand_landmarks[:2]:
        landmark_feats.append(normalize_hand(hand))
    while len(landmark_feats) < 2:
        landmark_feats.append(np.zeros(42))
    landmarks = np.concatenate(landmark_feats).astype(np.float32)  # (84,)

    # Crop image around first detected hand
    x1, y1, x2, y2 = get_bounding_box(results.hand_landmarks[0], w, h)
    crop = img_rgb[y1:y2, x1:x2]
    if crop.size == 0:
        return None, None

    crop = cv2.resize(crop, IMAGE_SIZE)
    crop = crop.astype(np.float32)
    crop = keras.applications.mobilenet_v2.preprocess_input(crop)  # scale to [-1, 1]

    return crop, landmarks

In [7]:
def build_dataset(hf_dataset):
    images, landmarks_list, labels = [], [], []
    for sample in tqdm(hf_dataset, desc="Extracting features"):
        crop, lms = extract_model_inputs(sample['image'])
        if crop is None:
            continue

        label = sample['label']
        label_id = label

        images.append(crop)
        landmarks_list.append(lms)
        labels.append(label_id)

    return (
        np.array(images),
        np.array(landmarks_list),
        np.array(labels),
        label_to_id
    )


In [8]:
def build_model(num_classes, num_landmarks=84):
    image_input = keras.Input(shape=(*IMAGE_SIZE, 3), name="image")
    backbone = keras.applications.MobileNetV2(
        input_shape=(*IMAGE_SIZE, 3),
        include_top=False,
        weights="imagenet"
    )
    backbone.trainable = False

    x_img = backbone(image_input, training=False)
    x_img = layers.GlobalAveragePooling2D()(x_img)
    x_img = layers.Dense(256, activation='relu')(x_img)
    x_img = layers.BatchNormalization()(x_img)
    x_img = layers.Dropout(0.3)(x_img)

    # --- Landmark stream ---
    landmark_input = keras.Input(shape=(num_landmarks,), name="landmarks")
    x_lm = layers.Dense(256, activation='relu')(landmark_input)
    x_lm = layers.BatchNormalization()(x_lm)
    x_lm = layers.Dropout(0.3)(x_lm)
    x_lm = layers.Dense(256, activation='relu')(x_lm)
    x_lm = layers.BatchNormalization()(x_lm)

    # --- Fusion ---
    fused = layers.Concatenate()([x_img, x_lm])
    x = layers.Dense(256, activation='relu')(fused)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation='relu')(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return keras.Model(
        inputs={"image": image_input, "landmarks": landmark_input},
        outputs=outputs
    ), backbone


In [15]:
X_images, X_landmarks, y, label_to_id = build_dataset(dataset)
num_classes = len(label_to_id)
print(f"\nDataset: {len(y)} samples, {num_classes} classes")
print(f"Image shape: {X_images.shape}, Landmark shape: {X_landmarks.shape}")

Extracting features: 100%|██████████| 14827/14827 [03:40<00:00, 67.11it/s]



Dataset: 11036 samples, 28 classes
Image shape: (11036, 224, 224, 3), Landmark shape: (11036, 84)


In [16]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_img_train, X_img_val, X_lm_train, X_lm_val, y_train, y_val = train_test_split(
    X_images, X_landmarks, y_encoded,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [17]:
y_train

array([23, 26, 19, ...,  2,  5,  6], shape=(8828,))

In [18]:
# Class weights for imbalanced data
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights))

In [19]:
model, backbone = build_model(num_classes)
model.compile(
    optimizer=keras.optimizers.Adam(LEARNING_RATE),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)
model.summary()

# Callbacks
callbacks = [
    keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=4, min_lr=1e-6),
    keras.callbacks.ModelCheckpoint("best_model_mobilenet.keras", save_best_only=True)
]

# Phase 1: Train with frozen backbone
print("\nPhase 1: Training with frozen backbone...")
model.fit(
    {"image": X_img_train, "landmarks": X_lm_train},
    y_train,
    validation_data=({"image": X_img_val, "landmarks": X_lm_val}, y_val),
    epochs=FINE_TUNE_EPOCH_START,
    batch_size=BATCH_SIZE,
    class_weight=class_weight_dict,
    callbacks=callbacks
)

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image (InputLayer)  │ (None, 224, 224,  │          0 │ -                 │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ landmarks           │ (None, 84)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mobilenetv2_1.00_2… │ (None, 7, 7,      │  2,257,984 │ image[0][0]       │
│ (Functional)        │ 1280)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 256)       │     21,760 │ landmarks[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 1280)      │          0 │ mobilenetv2_1.00… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256)       │      1,024 │ dense_7[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 256)       │    327,936 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 256)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256)       │      1,024 │ dense_6[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 256)       │     65,792 │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 256)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256)       │      1,024 │ dense_8[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 512)       │          0 │ dropout_3[0][0],  │
│ (Concatenate)       │                   │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 256)       │    131,328 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256)       │      1,024 │ dense_9[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 256)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 128)       │     32,896 │ dropout_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 28)        │      3,612 │ dense_10[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,845,404 (10.85 MB)

 Trainable params: 585,372 (2.23 MB)

 Non-trainable params: 2,260,032 (8.62 MB)


Phase 1: Training with frozen backbone...
Epoch 1/10
276/276 ━━━━━━━━━━━━━━━━━━━━ 56s 190ms/step - accuracy: 0.7288 - loss: 0.9501 - val_accuracy: 0.8777 - val_loss: 0.4366 - learning_rate: 0.0010
Epoch 2/10
276/276 ━━━━━━━━━━━━━━━━━━━━ 51s 183ms/step - accuracy: 0.9068 - loss: 0.2889 - val_accuracy: 0.9438 - val_loss: 0.1860 - learning_rate: 0.0010
Epoch 3/10
276/276 ━━━━━━━━━━━━━━━━━━━━ 50s 182ms/step - accuracy: 0.9357 - loss: 0.2023 - val_accuracy: 0.9407 - val_loss: 0.2005 - learning_rate: 0.0010
Epoch 4/10
276/276 ━━━━━━━━━━━━━━━━━━━━ 49s 178ms/step - accuracy: 0.9486 - loss: 0.1547 - val_accuracy: 0.9642 - val_loss: 0.1228 - learning_rate: 0.0010
Epoch 5/10
276/276 ━━━━━━━━━━━━━━━━━━━━ 49s 179ms/step - accuracy: 0.9566 - loss: 0.1244 - val_accuracy: 0.9611 - val_loss: 0.1419 - learning_rate: 0.0010
Epoch 6/10
276/276 ━━━━━━━━━━━━━━━━━━━━ 49s 179ms/step - accuracy: 0.9596 - loss: 0.1187 - val_accuracy: 0.9638 - val_loss: 0.1354 - learning_rate: 0.0010
Epoch 7/10
276/276 ━━━━━━━━

In [21]:
# Phase 2: Fine-tune top layers of backbone
print("\nPhase 2: Fine-tuning backbone...")
backbone.trainable = True
for layer in backbone.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(FINE_TUNE_LR),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

model.fit(
    {"image": X_img_train, "landmarks": X_lm_train},
    y_train,
    validation_data=({"image": X_img_val, "landmarks": X_lm_val}, y_val),
    epochs=EPOCHS,
    initial_epoch=FINE_TUNE_EPOCH_START,
    batch_size=BATCH_SIZE,
    class_weight=class_weight_dict,
    callbacks=callbacks
)

print("\nDone. Best model saved to best_model_mobilenet.keras")


Phase 2: Fine-tuning backbone...
Epoch 11/50
276/276 ━━━━━━━━━━━━━━━━━━━━ 65s 219ms/step - accuracy: 0.8324 - loss: 0.6079 - val_accuracy: 0.9135 - val_loss: 0.3184 - learning_rate: 1.0000e-05
Epoch 12/50
276/276 ━━━━━━━━━━━━━━━━━━━━ 58s 211ms/step - accuracy: 0.9135 - loss: 0.2758 - val_accuracy: 0.9420 - val_loss: 0.2189 - learning_rate: 1.0000e-05
Epoch 13/50
276/276 ━━━━━━━━━━━━━━━━━━━━ 58s 209ms/step - accuracy: 0.9444 - loss: 0.1644 - val_accuracy: 0.9606 - val_loss: 0.1461 - learning_rate: 1.0000e-05
Epoch 14/50
276/276 ━━━━━━━━━━━━━━━━━━━━ 58s 209ms/step - accuracy: 0.9568 - loss: 0.1293 - val_accuracy: 0.9674 - val_loss: 0.1357 - learning_rate: 1.0000e-05
Epoch 15/50
276/276 ━━━━━━━━━━━━━━━━━━━━ 58s 209ms/step - accuracy: 0.9651 - loss: 0.0946 - val_accuracy: 0.9710 - val_loss: 0.1236 - learning_rate: 5.0000e-06
Epoch 16/50
276/276 ━━━━━━━━━━━━━━━━━━━━ 58s 209ms/step - accuracy: 0.9703 - loss: 0.0876 - val_accuracy: 0.9728 - val_loss: 0.1114 - learning_rate: 5.0000e-06
Epoch 

In [22]:
import json
with open("mn_training_labels.json", "w") as f:
    json.dump(label_to_id, f)